In [ ]:
import argparse
import subprocess
import os
import sys
import glob
import numpy as np

In [ ]:
def run_cmd(cmd):
    print(f"\n=> Running: {' '.join(cmd)}")
    try:
        subprocess.run(cmd, check=True)
    except subprocess.CalledProcessError as e:
        print(f"Command failed with exit code {e.returncode}")
        sys.exit(e.returncode)

In [ ]:
if __name__ == "__main__":
    parser = argparse.ArgumentParser(description="Master Pipeline Orchestrator")
    parser.add_argument("--mode", choices=['quick', 'full'], required=True, 
                        help="'quick' for Phase 2 on existing checkpoint, 'full' for Phase 1b then Phase 2")
    parser.add_argument("--max-radius", type=int, default=15, help="Max Hamming radius for Z3")
    parser.add_argument("--timeout", type=int, default=300000, help="Z3 timeout in ms")
    parser.add_argument("--iterations", type=int, default=25_000_000, help="Phase 1b iterations per chain")
    args = parser.parse_args()
    
    script_dir = os.path.dirname(os.path.abspath(__file__))
    
    if args.mode == 'quick':
        print("=== RUNNING QUICK PIPELINE ===")
        ckpt_path = os.path.join(script_dir, "sa_checkpoint.npz")
        cmd = [sys.executable, os.path.join(script_dir, "phase_2b_z3_sweep.py"), 
               "--checkpoint", ckpt_path, 
               "--max-radius", str(args.max_radius), 
               "--timeout", str(args.timeout)]
        run_cmd(cmd)
        
    elif args.mode == 'full':
        print("=== RUNNING FULL PIPELINE ===")
        print("Phase 1: Multi-chain Simulated Annealing")
        cmd1 = [sys.executable, os.path.join(script_dir, "phase_1b_multichain_sa.py"), 
                "--iterations", str(args.iterations),
                "--out-dir", os.path.join(script_dir, "sa_runs")]
        run_cmd(cmd1)
        
        print("\nPhase 2: Z3 Sweep on top 5 checkpoints")
        ckpts = glob.glob(os.path.join(script_dir, "sa_runs", "sa_checkpoint_drop*.npz"))
        ckpt_errors = []
        for c in ckpts:
            data = np.load(c)
            ckpt_errors.append((c, int(data['error'])))
            
        ckpt_errors.sort(key=lambda x: x[1])
        top_5 = ckpt_errors[:5]
        
        for c, err in top_5:
            print(f"\nEvaluating checkpoint {c} with error {err}/729...")
            cmd2 = [sys.executable, os.path.join(script_dir, "phase_2b_z3_sweep.py"), 
                   "--checkpoint", c, 
                   "--max-radius", str(args.max_radius), 
                   "--timeout", str(args.timeout)]
            try:
                subprocess.run(cmd2, check=True)
            except subprocess.CalledProcessError:
                print(f"Sweep for {c} finished (possibly UNSAT or TIMEOUT).")
                
        print("\nFull pipeline completed.")

In [ ]:
!python3 run_full_pipeline.py --mode full